# Wound-healing analyses and paper figures

This notebook generates the wound-healing statistical summaries and figures used for the manuscript. It starts from the processed, untransformed ear-hole phenotype table produced by `Preprocessing.ipynb`.

The notebook is organized into: (1) setup and data loading, (2) strain effects in wild-type focal mice, (3) genotype/sex model comparisons, (4) stratified genotype-effect figures, and (5) broader model-selection analyses.


## 1. Setup

Define project-relative paths once here. The notebook assumes it is run from the repository root (or that `PROJECT_ROOT` is changed accordingly).


In [88]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu
from scipy.stats import shapiro, kstest
from scipy.stats import levene
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from pathlib import Path
import re
from scipy.stats import boxcox
import os


In [ ]:
# Project-relative paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures" / "wound_healing"
STATS_DIR = RESULTS_DIR / "statistics"

EAR_HOLE_FILE = PROCESSED_DATA_DIR / "ear_hole_area_untransformed.xlsx"
COMBINED_ANOVA_FILE = STATS_DIR / "Combined_ANOVA_only.xlsx"

FIG_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Phenotype input: {EAR_HOLE_FILE}")
print(f"Figure output directory: {FIG_DIR}")


## 2. Load processed ear-hole phenotype data

The input is the untransformed phenotype table generated by the wound-healing preprocessing notebook. Zero-valued areas are replaced with `0.01` before Box–Cox transformation, matching the original analysis workflow.


In [ ]:
ear_hole_area = pd.read_excel(EAR_HOLE_FILE)
ear_hole_area['Area'] = ear_hole_area['Area'].replace(0, 0.01)


## 3. Strain differences in wild-type focal mice

Restrict the dataset to wild-type focal mice, apply the Box–Cox transformation used for this comparison, and test for strain differences using one-way ANOVA followed by Tukey pairwise comparisons. The accompanying figure is saved in both SVG and PDF format.


In [ ]:
ear_hole_area_wt = ear_hole_area[ear_hole_area['Genotype'] == 'WT'].copy()


In [ ]:
transformed_area, lam = boxcox(ear_hole_area_wt['Area'])
ear_hole_area_wt['Area_boxcox'] = transformed_area


In [ ]:
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Data ──────────────────────────────────────────────────────────────────────
strain_order = ['MRL', 'DBA', 'CBA', 'B6']
strain_labels = ['MRL/MpJ', 'DBA/2J', 'CBA/J', 'C57BL/6J']
combined_df = ear_hole_area_wt

# ── Statistics ────────────────────────────────────────────────────────────────
groups = [
    combined_df.loc[combined_df['COT'] == s, 'Area_boxcox']
    for s in strain_order
]
f_stat, p_value = f_oneway(*groups)
n_total = combined_df['Area_boxcox'].notna().sum()
n_groups = len(strain_order)
df_between = n_groups - 1
print(f"One-way ANOVA: F({df_between}, {n_total - n_groups}) = {f_stat:.4f}, p = {p_value:.4e}")

tukey = pairwise_tukeyhsd(
    endog=combined_df['Area_boxcox'],
    groups=combined_df['COT'],
    alpha=0.05
)

# ── Figure ────────────────────────────────────────────────────────────────────
FONT = 'Arial'
plt.rcParams.update({
    'font.family': FONT,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.labelweight': 'bold',
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
})

palette = ['#6BAF92', '#E8956D', '#8FA8C8', '#C490B8']  # muted, journal-friendly

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=combined_df,
    x='COT',
    y='Area_boxcox',
    order=strain_order,
    palette=palette,
    width=0.5,
    linewidth=0.8,
    fliersize=0,          # hide default outlier markers — raw dots cover this
    boxprops=dict(linewidth=0.8, alpha=0.8),
    medianprops=dict(color='black', linewidth=1.5),
    whiskerprops=dict(linewidth=0.8),
    capprops=dict(linewidth=0.8),
    ax=ax
)

sns.stripplot(
    data=combined_df,
    x='COT',
    y='Area_boxcox',
    color='black',
    size=2.5,
    order=strain_order,
    jitter=0.08,
    alpha=0.5,
    ax=ax
)

# ── Axes formatting ───────────────────────────────────────────────────────────
ax.set_xlabel("Strain", labelpad=8)
ax.set_ylabel("Ear hole area", labelpad=8)
ax.set_title('')
ax.set_xticklabels(strain_labels)
sns.despine(ax=ax, top=True, right=True)
ax.yaxis.set_tick_params(length=4)
ax.xaxis.set_tick_params(length=4)
ymax = combined_df['Area_boxcox'].max()
ax.set_ylim(top=ymax * 1.15)

# ── ANOVA annotation (Option B) ───────────────────────────────────────────────
p_str = f"p < 0.001" if p_value < 0.001 else f"p = {p_value:.3f}"
anova_text = (
    f"One-way ANOVA\n"
    f"F({df_between}, {n_total - n_groups}) = {f_stat:.2f}, {p_str}"
)
ax.text(
    0.97, 0.97, anova_text,
    transform=ax.transAxes,
    fontsize=9,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='None', linewidth=0.6)
)

plt.tight_layout()

# ── Save figure ───────────────────────────────────────────────────────────────
fig.savefig(FIG_DIR / 'wound_healing_by_strain.svg', format='svg', dpi=600, bbox_inches='tight')
fig.savefig(FIG_DIR / 'wound_healing_by_strain.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
print("Figure saved.")

# ── Tukey HSD → xlsx ─────────────────────────────────────────────────────────
tukey_df = pd.DataFrame(
    data=tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)

# Rename columns to be clean
tukey_df.columns = ['Group 1', 'Group 2', 'Mean diff', 'p-adj', 'Lower CI (95%)', 'Upper CI (95%)', 'Reject H₀']

# Map internal strain codes back to full names
name_map = dict(zip(strain_order, strain_labels))
tukey_df['Group 1'] = tukey_df['Group 1'].map(lambda x: name_map.get(x, x))
tukey_df['Group 2'] = tukey_df['Group 2'].map(lambda x: name_map.get(x, x))

# Round numeric columns
for col in ['Mean diff', 'p-adj', 'Lower CI (95%)', 'Upper CI (95%)']:
    tukey_df[col] = pd.to_numeric(tukey_df[col]).round(4)

# ── Build xlsx ────────────────────────────────────────────────────────────────
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Tukey HSD"

# Styles
header_fill = PatternFill('solid', start_color='2F5496', end_color='2F5496')
sig_fill    = PatternFill('solid', start_color='D9EAD3', end_color='D9EAD3')
alt_fill    = PatternFill('solid', start_color='F2F2F2', end_color='F2F2F2')
header_font = Font(name=FONT, bold=True, color='FFFFFF', size=11)
body_font   = Font(name=FONT, size=10)
center_align = Alignment(horizontal='center', vertical='center')
left_align   = Alignment(horizontal='left',   vertical='center')
thin_border  = Border(
    bottom=Side(style='thin', color='CCCCCC'),
    top=Side(style='thin', color='CCCCCC')
)

# Title row
ws.merge_cells('A1:G1')
title_cell = ws['A1']
title_cell.value = "Tukey HSD Post-hoc Test — Immobility Time by Strain"
title_cell.font  = Font(name=FONT, bold=True, size=12)
title_cell.alignment = left_align
ws.row_dimensions[1].height = 22

# ANOVA summary row
ws.merge_cells('A2:G2')
anova_cell = ws['A2']
anova_cell.value = f"One-way ANOVA: F({df_between}, {n_total - n_groups}) = {f_stat:.4f}, {p_str}"
anova_cell.font  = Font(name=FONT, italic=True, size=10, color='444444')
anova_cell.alignment = left_align
ws.row_dimensions[2].height = 18

ws.row_dimensions[3].height = 6  # spacer

# Header row (row 4)
headers = list(tukey_df.columns)
for col_idx, h in enumerate(headers, start=1):
    cell = ws.cell(row=4, column=col_idx, value=h)
    cell.font      = header_font
    cell.fill      = header_fill
    cell.alignment = center_align
ws.row_dimensions[4].height = 20

# Data rows
for row_idx, row in tukey_df.iterrows():
    excel_row = row_idx + 5
    is_sig = bool(row['Reject H₀'])
    for col_idx, val in enumerate(row, start=1):
        cell = ws.cell(row=excel_row, column=col_idx, value=val)
        cell.font = body_font
        cell.fill = sig_fill if is_sig else (alt_fill if row_idx % 2 == 0 else PatternFill())
        cell.alignment = center_align if col_idx > 2 else left_align
        cell.border = thin_border
    ws.row_dimensions[excel_row].height = 18

# Footnote
footnote_row = len(tukey_df) + 6
ws.merge_cells(f'A{footnote_row}:G{footnote_row}')
fn = ws.cell(row=footnote_row, column=1,
             value="Green rows indicate significant differences (p-adj < 0.05). CI = confidence interval.")
fn.font = Font(name=FONT, italic=True, size=9, color='666666')
fn.alignment = left_align

# Column widths
col_widths = [14, 14, 14, 12, 16, 16, 14]
for i, w in enumerate(col_widths, start=1):
    ws.column_dimensions[get_column_letter(i)].width = w

#output_path = FIG_DIR / 'tukey_hsd_wound_healing_strain_first.xlsx'
#wb.save(output_path)
#print(f"Tukey table saved → {output_path}")


## 4. Transform the full phenotype dataset and define strain subsets

A Box–Cox transformation is applied to the full dataset for the genotype/sex analyses below. The original strain-specific subsets are retained because subsequent model comparisons use them directly.


In [ ]:
transformed_area, lam = boxcox(ear_hole_area['Area'])
ear_hole_area['Area_boxcox'] = transformed_area


In [ ]:
ear_hole_area['Strain'] = ear_hole_area['COT']


In [ ]:
MRL = ear_hole_area[ear_hole_area['Strain'] == 'MRL']
DBA = ear_hole_area[ear_hole_area['Strain'] == 'DBA']
CBA = ear_hole_area[ear_hole_area['Strain'] == 'CBA']
B6 = ear_hole_area[ear_hole_area['Strain'] == 'B6']


## 5. Sex and genotype model comparison within DBA/2J

Fit the original OLS model set for the DBA/2J subset, compare AIC values, and calculate nested-model ANOVA comparisons. The resulting AIC and ANOVA tables are written to `results/statistics/`.


In [ ]:
import statsmodels.api as sm
model_data = DBA.copy()

m0 = smf.ols("Area_boxcox ~ 1", data=model_data).fit()

m1 = smf.ols("Area_boxcox ~ Sex", data=model_data).fit()

m2 = smf.ols("Area_boxcox ~ Sex + Genotype", data=model_data).fit()

m3 = smf.ols("Area_boxcox ~ Sex * Genotype", data=model_data).fit()

m4 = smf.ols("Area_boxcox ~ Genotype", data=model_data).fit()

aic_values = {
    "m0 (null)": m0.aic,
    "m1 (Sex)": m1.aic,
    "m2 (Sex + Genotype)": m2.aic,
    "m3 (Sex * Genotype)": m3.aic,
    "m4 (Genotype)": m4.aic
}

for name, val in aic_values.items():
    print(f"{name}: {val:.2f}")

aic_df = pd.DataFrame({
"Model": list(aic_values.keys()),
"AIC": list(aic_values.values())
})

aic_df["ΔAIC"] = aic_df["AIC"] - aic_df["AIC"].min()
print(aic_df)


anova_res2 = sm.stats.anova_lm(m1, m3)
print(anova_res2)



## 6. Ear-hole area by strain, sex, and genotype

Generate the manuscript boxplot stratified by strain, sex, and genotype. Significance annotations are read from the combined likelihood-ratio-test results table specified by `COMBINED_ANOVA_FILE`.


## 7. Sex-stratified genotype comparisons

Calculate WT-versus-HET comparisons within each strain-by-sex subgroup and prepare the stratified genotype-effect plot used to inspect context-specific effects.


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# -----------------------------
# User settings
# -----------------------------
df = ear_hole_area.copy()

strain_order = ["MRL", "DBA", "CBA", "B6"]
sex_order = ["M", "F"]
genotype_order = ["WT", "HET"]

phenotype_col = "Area_boxcox"

output_stats_file = STATS_DIR / "ttests_by_strain_sex.csv"
output_plot_file = FIG_DIR / "combined_barplot_16bars.png"


# -----------------------------
# Ensure categorical ordering
# -----------------------------
df["Strain"] = pd.Categorical(df["Strain"], categories=strain_order, ordered=True)
df["Sex"] = pd.Categorical(df["Sex"], categories=sex_order, ordered=True)
df["Genotype"] = pd.Categorical(df["Genotype"], categories=genotype_order, ordered=True)


# -----------------------------
# Welch t-tests: WT vs HET within each Strain x Sex subgroup
# -----------------------------
ttest_results = []

for strain in strain_order:
    for sex in sex_order:
        sub = df[(df["Strain"] == strain) & (df["Sex"] == sex)]

        wt = sub.loc[sub["Genotype"] == "WT", phenotype_col].dropna()
        het = sub.loc[sub["Genotype"] == "HET", phenotype_col].dropna()

        if len(wt) >= 2 and len(het) >= 2:
            t_stat, p_val = stats.ttest_ind(wt, het, equal_var=True)

            # Welch-Satterthwaite degrees of freedom
            s1, s2 = wt.var(ddof=1), het.var(ddof=1)
            n1, n2 = len(wt), len(het)

            numerator = (s1 / n1 + s2 / n2) ** 2
            denominator = ((s1 / n1) ** 2 / (n1 - 1)) + ((s2 / n2) ** 2 / (n2 - 1))
            df_welch = numerator / denominator

        else:
            t_stat, p_val, df_welch = np.nan, np.nan, np.nan

        ttest_results.append({
            "Strain": strain,
            "Sex": sex,
            "Comparison": "WT vs HET",
            "WT_n": len(wt),
            "HET_n": len(het),
            "WT_mean": wt.mean(),
            "HET_mean": het.mean(),
            "WT_sem": wt.sem(),
            "HET_sem": het.sem(),
            "t_stat": t_stat,
            "df": df_welch,
            "p_value": p_val
        })

ttest_df = pd.DataFrame(ttest_results)

# Optional: multiple-testing correction across the 8 Welch tests
# Bonferroni correction across 2 sexes x 3 phenotypes = 6 tests
bonferroni_n_tests = 2

ttest_df["p_bonferroni_sex_phenotype"] = np.minimum(
    ttest_df["p_value"] * bonferroni_n_tests,
    1.0
)

#ttest_df.to_csv(output_stats_file, index=False)

print(ttest_df)
print(f"Saved Welch t-test results to: {output_stats_file}")


## 8. Model selection for strain/background, sex, and genotype effects

Fit the original OLS model set used to compare strain-, sex-, genotype-, and genetic-background-dependent effects. AIC values are collected into a single table for model comparison.


In [ ]:
model_data = ear_hole_area.copy()

# ============================================================
# Add Background column
# ============================================================

model_data["Background"] = np.where(
    model_data["Strain"] == "B6",
    "Same",
    "Mixed"
)

# Optional: make ordering explicit
model_data["Background"] = pd.Categorical(
    model_data["Background"],
    categories=["Same", "Mixed"]
)

print(model_data[["Strain", "Background"]].value_counts())


# ============================================================
# Original OLS models using Strain
# ============================================================

m0 = smf.ols(
    "Area_boxcox ~ 1",
    data=model_data
).fit()

m1 = smf.ols(
    "Area_boxcox ~ Strain",
    data=model_data
).fit()

m2 = smf.ols(
    "Area_boxcox ~ Sex",
    data=model_data
).fit()

m3 = smf.ols(
    "Area_boxcox ~ Strain + Sex",
    data=model_data
).fit()

m4 = smf.ols(
    "Area_boxcox ~ Strain * Sex",
    data=model_data
).fit()

m5 = smf.ols(
    "Area_boxcox ~ Strain + Sex + Genotype",
    data=model_data
).fit()

m6 = smf.ols(
    "Area_boxcox ~ Strain + Sex * Genotype",
    data=model_data
).fit()

m7 = smf.ols(
    "Area_boxcox ~ Strain * Genotype + Sex",
    data=model_data
).fit()

m8 = smf.ols(
    "Area_boxcox ~ Strain * Sex + Genotype",
    data=model_data
).fit()

m9 = smf.ols(
    "Area_boxcox ~ Strain * Sex * Genotype",
    data=model_data
).fit()


# ============================================================
# Parallel models using Background instead of Strain
# ============================================================

# Equivalent of m1
m10 = smf.ols(
    "Area_boxcox ~ Strain +  Sex + Genotype + Background:Genotype",
    data=model_data
).fit()

# Equivalent of m3
m11 = smf.ols(
    "Area_boxcox ~ Strain + Sex + Genotype + Strain:Sex + Sex:Genotype + Background:Genotype + Sex:Background:Genotype",
    data=model_data
).fit()

# ============================================================
# AIC comparison
# ============================================================

aic_values = {

    # Shared models
    "m0 (null)": m0.aic,
    "m2 (Sex)": m2.aic,

    # Strain models
    "m1 (Strain)": m1.aic,
    "m3 (Strain + Sex)": m3.aic,
    "m4 (Strain * Sex)": m4.aic,
    "m5 (Strain + Sex + Genotype)": m5.aic,
    "m6 (Strain + Sex * Genotype)": m6.aic,
    "m7 (Strain * Genotype + Sex)": m7.aic,
    "m8 (Strain * Sex + Genotype)": m8.aic,
    "m9 (Strain * Sex * Genotype)": m9.aic,

    # Background models
    "m10 (Strain + Sex + Genotype + SameBackground:Genotype)": m10.aic,
    "m11 (Strain + Sex + Genotype + Strain:Sex + Sex:Genotype + Background:Genotype + Sex:Background:Genotype)": m11.aic
}


# ============================================================
# Make AIC dataframe
# ============================================================

aic_df = pd.DataFrame({
    "Model": list(aic_values.keys()),
    "AIC": list(aic_values.values())
})

aic_df["ΔAIC"] = aic_df["AIC"] - aic_df["AIC"].min()

# Sort best model first
aic_df = aic_df.sort_values(
    "AIC",
    ascending=True
).reset_index(drop=True)

print(aic_df)
